# Detección de eventos en spreads cointegrados

**Autor:** Samuel  
**Institución:** Tecnológico de Monterrey  
**Área:** Pairs trading en Forex  
**Notebooks previos:** Backtest-Corregido, Beta-Anclado-OU, Remuestreo-Costos, Amplitud

En *Amplitud* encontramos una señal bruta pequeña pero real, de tamaño similar a los costos. Aquí estudiamos **dónde se pierde**: si las pérdidas se concentran en pocas operaciones en las que la relación entre monedas se rompe, un detector de rupturas podría mejorar el neto.

## Estructura de este Notebook
1. Fase 0: Motor y datos (se reutiliza el de *Amplitud*)
2. Fase 1: Operaciones individuales
3. Fase 2: Techo teórico (qué pasaría si evitáramos las rupturas)
4. Fase 3: Detector de rupturas (CUSUM)
5. Fase 4: Validación con placebo
6. Fase 5: Filtro horario
7. Conclusión

## Fase 0: Motor y Datos

Reutilizamos los datos y algoritmos probados en 'Amplitud.ipynb'

In [ ]:
# Fase 1 — Datos
# %pip install yfinance pandas numpy

# Fase 3 — Cointegración y corrección por pruebas múltiples
# %pip install scipy statsmodels

# Fase 4 — Motor (opcional, acelera ~50x)
# %pip install numba

In [11]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
from itertools import combinations
from scipy import stats

In [3]:
pd.set_option('display.max_columns', None)

base = ['EURUSD','GBPUSD','AUDUSD','NZDUSD','USDJPY','USDCAD',
        'USDCHF','USDMXN','USDCNY','USDSEK','USDNOK','USDSGD']
nuevas = ['USDZAR','USDBRL','USDINR','USDKRW','USDPLN','USDHUF','USDCZK','USDILS','USDTHB']

tickers = {k: f'{k}=X' for k in base + nuevas}

raw_close = yf.download(list(tickers.values()), period='729d', interval='60m', progress=False)['Close']
ticker_to_name = {v: k for k, v in tickers.items()}
raw_data = raw_close.rename(columns=ticker_to_name)

print(f"Descargado: {raw_data.shape}")

Descargado: (17419, 21)


In [4]:
# Cobertura: fracción de barras horarias con dato, respecto a la divisa con más datos
cobertura = raw_data.notna().mean() / raw_data.notna().mean().max()
print(cobertura.sort_values().round(3))

MIN_COBERTURA = 0.90
aceptadas = cobertura[cobertura >= MIN_COBERTURA].index.tolist()
descartadas = [c for c in tickers if c not in aceptadas]
print(f"\nAceptadas ({len(aceptadas)}): {aceptadas}")
print(f"Descartadas por cobertura ({len(descartadas)}): {descartadas}")

Ticker
USDBRL    0.681
USDCNY    0.772
USDINR    0.817
USDKRW    0.951
USDILS    0.965
USDSEK    0.977
USDTHB    0.985
USDNOK    0.989
USDJPY    0.989
USDCHF    0.991
USDSGD    0.994
EURUSD    0.994
GBPUSD    0.994
USDMXN    0.995
USDZAR    0.998
AUDUSD    0.998
NZDUSD    0.998
USDPLN    0.998
USDCAD    0.998
USDHUF    1.000
USDCZK    1.000
dtype: float64

Aceptadas (18): ['AUDUSD', 'EURUSD', 'GBPUSD', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDCZK', 'USDHUF', 'USDILS', 'USDJPY', 'USDKRW', 'USDMXN', 'USDNOK', 'USDPLN', 'USDSEK', 'USDSGD', 'USDTHB', 'USDZAR']
Descartadas por cobertura (3): ['USDCNY', 'USDBRL', 'USDINR']


In [5]:
data = raw_data[aceptadas].ffill().dropna()

# Rangos aproximados — ajústalos si alguna divisa queda fuera por una razón legítima
expected = {'EURUSD':(0.9,1.4),'GBPUSD':(1.0,1.6),'AUDUSD':(0.5,0.9),'NZDUSD':(0.5,0.8),
            'USDJPY':(100,175),'USDCAD':(1.1,1.6),'USDCHF':(0.7,1.1),'USDMXN':(14,26),
            'USDCNY':(6.3,7.6),'USDSEK':(8,12),'USDNOK':(8,12.5),'USDSGD':(1.1,1.6),
            'USDZAR':(12,24),'USDBRL':(4,7),'USDINR':(75,100),'USDKRW':(1100,1700),
            'USDPLN':(3.3,4.8),'USDHUF':(280,430),'USDCZK':(19,27),'USDILS':(3,4.3),'USDTHB':(28,42)}

chk = pd.DataFrame({'media': data.mean(), 'min': data.min(), 'max': data.max()})
chk['ok'] = [expected[c][0] <= chk.loc[c, 'media'] <= expected[c][1] for c in chk.index]
display(chk.round(4))
print(data.shape, data.index.min(), '->', data.index.max())
assert chk['ok'].all(), "Algún nivel de precio no corresponde a su nombre (o el rango esperado hay que ajustarlo)"

,media,min,max,ok
Ticker,,,,
AUDUSD,0.6672,0.5952,0.7273,True
EURUSD,1.1216,1.0196,1.2047,True
GBPUSD,1.3118,1.2108,1.3850,True
NZDUSD,0.5903,0.5512,0.6376,True
USDCAD,1.3855,1.3352,1.4781,True
USDCHF,0.8390,0.7609,0.9222,True
USDCZK,22.0900,20.1080,24.7250,True
USDHUF,347.2892,300.1800,406.1000,True
USDILS,3.4147,2.7993,3.8435,True


(17419, 18) 2024-01-11 00:00:00+00:00 -> 2026-10-27 16:00:00+00:00


In [7]:
nombres = {
    'EURUSD': 'Euro',        'GBPUSD': 'Libra',       'AUDUSD': 'Dolar_AUS',   'NZDUSD': 'Dolar_NZ',
    'USDCAD': 'Dolar_CAN',   'USDCHF': 'Franco_SUI',  'USDCZK': 'Corona_CZ',   'USDHUF': 'Forinto_HU',
    'USDILS': 'Shekel_IL',   'USDJPY': 'Yen',         'USDKRW': 'Won_KR',      'USDMXN': 'Peso_MX',
    'USDNOK': 'Corona_NO',   'USDPLN': 'Zloty_PL',    'USDSEK': 'Corona_SE',   'USDSGD': 'Dolar_SG',
    'USDTHB': 'Baht_TH',     'USDZAR': 'Rand_ZA',
}
# Documenta la dirección de cada cotización (no cambia los datos)
cotizacion = {nombres[t]: ('USD por unidad' if t.endswith('USD') else 'unidades por USD') for t in nombres}

data_t = data.copy()          # versión con tickers, por si hay que compararla con los notebooks previos
data = data_t.rename(columns=nombres)
print(pd.Series(cotizacion))

Euro            USD por unidad
Libra           USD por unidad
Dolar_AUS       USD por unidad
Dolar_NZ        USD por unidad
Dolar_CAN     unidades por USD
Franco_SUI    unidades por USD
Corona_CZ     unidades por USD
Forinto_HU    unidades por USD
Shekel_IL     unidades por USD
Yen           unidades por USD
Won_KR        unidades por USD
Peso_MX       unidades por USD
Corona_NO     unidades por USD
Zloty_PL      unidades por USD
Corona_SE     unidades por USD
Dolar_SG      unidades por USD
Baht_TH       unidades por USD
Rand_ZA       unidades por USD
dtype: str


In [10]:
warnings.filterwarnings('ignore')
try:
    from numba import njit          # opcional: %pip install numba (acelera mucho)
except ImportError:
    njit = lambda f: f

In [12]:
TEST_DAYS, TRAIN_DAYS, MIN_TRAIN = 30, 120, 240   # ventanas de 30 días, entrenamiento de 120
STOP, HL_MAX = 3.5, 24 * 15                       # stop en sigmas; vida media máxima (horas)
COST_BPS = 1.0                                    # por pata y por lado
N_MAX, JACC_MAX = 20, 0.4                         # canastas por ventana; traslape máximo entre canastas

In [13]:
def fit_ou(s):
    x, y = s[:-1], s[1:]
    b, a = np.polyfit(x, y, 1)
    if not (0.0 < b < 1.0): return None
    r = y - (a + b * x)
    return dict(b=b, mu=a / (1 - b), sig=r.std(ddof=2) / np.sqrt(1 - b * b), hl=np.log(2) / (-np.log(b)))

@njit
def positions_from_z(z, k, e, stop):
    n = len(z); pos = np.zeros(n); p = 0.0; blocked = 0.0
    for t in range(n):
        zt = z[t]
        if blocked != 0.0 and abs(zt) < k: blocked = 0.0
        if p == 0.0:
            if blocked == 0.0:
                if zt >= k: p = -1.0
                elif zt <= -k: p = 1.0
        else:
            if p < 0 and zt <= e: p = 0.0
            elif p > 0 and zt >= -e: p = 0.0
            elif abs(zt) >= stop:
                blocked = 1.0 if zt > 0 else -1.0
                p = 0.0
        pos[t] = p
    return pos

def ventanas(index, train_days=TRAIN_DAYS):
    day = index.normalize().values; ud = np.unique(day)
    for j in range(MIN_TRAIN, len(ud) - TEST_DAYS + 1, TEST_DAYS):
        tr = (day >= ud[j - train_days]) & (day < ud[j]); te = (day >= ud[j]) & (day < ud[j + TEST_DAYS])
        yield j, tr, te

In [14]:
def _df(s, mu=None):
    sl = s[:-1] - (s[:-1].mean() if mu is None else mu); ds = np.diff(s)
    d = (sl @ ds) / (sl @ sl); e = ds - d * sl
    return -d, -(d / np.sqrt(e.var(ddof=2) / (sl @ sl)))

def puntaje_cestas(Ltr, cestas):
    sc = np.empty(len(cestas)); coefs = []
    for n, idx in enumerate(cestas):
        A = np.column_stack([np.ones(len(Ltr))] + [Ltr[:, m] for m in idx[1:]])
        coef = np.linalg.lstsq(A, Ltr[:, idx[0]], rcond=None)[0]
        sc[n] = _df(Ltr[:, idx[0]] - A @ coef)[1]
        coefs.append(coef)
    return sc, coefs

def elegir_cestas(sc, cestas, n_max=N_MAX, jacc_max=JACC_MAX):
    elegidas = []
    for n in np.argsort(-sc):
        s = set(cestas[n])
        if all(len(s & set(cestas[m])) / len(s | set(cestas[m])) <= jacc_max for m in elegidas):
            elegidas.append(n)
            if len(elegidas) == n_max: break
    return elegidas

def pesos_cesta(idx, coef, n_act):
    w = np.zeros(n_act); w[idx[0]] = 1.0
    for m, c in zip(idx[1:], coef[1:]): w[m] = -c
    return w

def preparar_seleccion(data, n_max=N_MAX, jacc_max=JACC_MAX, train_days=TRAIN_DAYS):
    Lp = np.log(data).values
    cestas = [c for k in (2, 3, 4) for c in combinations(range(data.shape[1]), k)]
    out = []
    for j, tr, te in ventanas(data.index, train_days):
        sc, coefs = puntaje_cestas(Lp[tr], cestas)
        sel = elegir_cestas(sc, cestas, n_max, jacc_max)
        W = np.column_stack([pesos_cesta(cestas[n], coefs[n], data.shape[1]) for n in sel])
        out.append(dict(window=j, tr=tr, te=te, W=W, cestas=[cestas[n] for n in sel], score=sc[sel]))
    return out

In [15]:
seleccion = preparar_seleccion(data)
print(len(seleccion), "ventanas;", sum(s['W'].shape[1] for s in seleccion), "canastas-ventana")

16 ventanas; 320 canastas-ventana


## Fase 1: Operaciones individuales

Hasta ahora medimos el promedio por ventana. Aquí registramos **cada operación** (entrada, duración, bruto, neto y cómo terminó) para ver qué la distingue de las demás.

Cada operación termina de una de tres formas:
- **revierte**: el spread volvió al promedio (lo esperado).
- **stop**: se alejó más de 3.5 sd y cortamos.
- **fin**: seguía abierta al terminar la ventana; se liquida a precio de mercado y **se cobra la salida** (el motor de *Amplitud* no lo cobraba).

In [16]:
def operaciones_spread(Lp_tr, Lp_te, w, idx_te, k=1.0, e=0.0, cost_bps=COST_BPS, retraso=1):
    """Lista de operaciones (centro fijo) de un spread, con bruto, neto y forma de salida."""
    s_tr, s_te = Lp_tr @ w, Lp_te @ w
    E = np.abs(w).sum()                                       # exposición bruta
    ou = fit_ou(s_tr)
    if ou is None or ou['hl'] > HL_MAX:
        return []
    z = (s_te - ou['mu']) / ou['sig']
    pos = positions_from_z(z, k, e, STOP)
    ds = np.diff(s_te); n = len(ds) - retraso
    ops, t = [], 0
    while t < n:
        if pos[t] == 0:
            t += 1; continue
        a = t
        while t < n and pos[t] == pos[a]:
            t += 1
        b = t
        bruto = (pos[a:b] * ds[a + retraso:b + retraso]).sum() / E
        if b >= n:
            salida = 'fin'
        elif abs(z[b]) >= STOP:
            salida = 'stop'
        else:
            salida = 'revierte'
        ops.append(dict(entrada=idx_te[a], horas=b - a, lado=int(pos[a]), bruto=bruto,
                        neto=bruto - 2 * cost_bps / 1e4, salida=salida))
    return ops

In [17]:
Lp = np.log(data).values
T = pd.DataFrame([dict(o, ventana=s['window'], cesta=i)
                  for s in seleccion for i in range(s['W'].shape[1])
                  for o in operaciones_spread(Lp[s['tr']], Lp[s['te']], s['W'][:, i], data.index[s['te']])])
print(len(T), "operaciones;  bruto total:", round(T.bruto.sum(), 4), " neto total:", round(T.neto.sum(), 4))

1164 operaciones;  bruto total: 0.208  neto total: -0.0248


In [18]:
display(T.groupby('salida').agg(n=('neto', 'size'), neto_medio=('neto', 'mean'),
                                neto_suma=('neto', 'sum'), horas=('horas', 'mean')).round(5))

,n,neto_medio,neto_suma,horas
salida,,,,
fin,97,-0.00103,-0.09995,118.95876
revierte,771,0.00141,1.08748,71.00908
stop,296,-0.00342,-1.01229,150.73649


## Fase 2: ¿Cuánto tendría que acertar un detector?

Clasificamos cada operación como **buena** (revierte) o **mala** (stop o fin). Si un detector evita una fracción *f* de las malas y, por error, una fracción *g* de las buenas, el neto cambia a: neto = (suma de buenas)(1 − g) − (pérdida de malas)(1 − f). Primero medimos qué combinaciones *(f, g)* serían rentables y luego revisamos si alguna característica **conocida al entrar** separa a las malas de las buenas.